# Model Lab

Compare TensorFlow/Keras HAR architectures while reusing the package preprocessing, subject-aware validation split, metrics, TFLite conversion, and host latency measurement.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from src.training.experiment_lab import run_keras_architecture

print('TensorFlow', tf.__version__)
print('Project root:', PROJECT_ROOT)

## Define A Candidate Model

Only edit this cell when trying a different architecture. Keep the input and output contract unchanged.

In [ ]:
def build_candidate_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape, name='imu_window')
    x = layers.SeparableConv1D(24, 5, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.AveragePooling1D(2)(x)
    x = layers.SeparableConv1D(32, 5, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='activity')(x)
    return keras.Model(inputs, outputs, name='notebook_ds_cnn')

## Run Experiment

This writes metrics, confusion matrix, Keras model, TFLite model, and a CSV summary under `outputs/lightweight/experiments/notebook_ds_cnn/`.

In [ ]:
result = run_keras_architecture(
    run_name='notebook_ds_cnn',
    model_builder=build_candidate_model,
    normalization='standard',
    epochs=20,
    patience=5,
    device='cpu',
)

{
    'accuracy': result['accuracy'],
    'macro_f1': result['macro_f1'],
    'weighted_f1': result['weighted_f1'],
    'params': result['model_info']['keras_params'],
    'tflite_size_bytes': result['model_info']['tflite_size_bytes'],
    'latency_ms_mean': result['model_info']['host_latency']['mean_ms'],
    'top_confusions': result['top_confusion_pairs'],
}

## Compare Runs

After trying several architectures, read the generated `summary.csv` files and compare accuracy, macro F1, TFLite size, and latency. Do not manually edit preprocessing or split logic inside the notebook.